# Block 04 – The router, on its own and in the graph

**You'll see:** the form the router fills in, its decisions and reasons, and the whole graph step by step.

In [ ]:
# Setup: run this cell first. It finds the repo, makes it importable and loads .env
import os, sys, pathlib

def find_repo_root():
    starts = []
    if "__vsc_ipynb_file__" in globals():                 # VS Code tells us where this notebook is
        starts.append(pathlib.Path(globals()["__vsc_ipynb_file__"]).resolve().parent)
    starts.append(pathlib.Path.cwd().resolve())
    for start in starts:
        for folder in [start, *start.parents]:
            if (folder / "common" / "config.py").exists():
                return folder
            if (folder / "agents-weekend" / "common" / "config.py").exists():
                return folder / "agents-weekend"
    raise RuntimeError(f"Can't find the agents-weekend repo from {starts}. "
                       "Open the agents-weekend folder in VS Code, or set ROOT by hand.")

ROOT = find_repo_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo:", ROOT)

from common.config import get_llm, get_cheap_llm, DEFAULT_MODEL, CHEAP_MODEL
from common.tracing import langfuse_config, trace_attributes, flush, tracing_enabled
print("main model:", DEFAULT_MODEL, "| cheap model:", CHEAP_MODEL, "| Langfuse:", tracing_enabled())

## 1. The router's form, as Claude sees it

In [ ]:
import json
from common.loader import load_block
b04 = load_block("04-multi-agent")
print(json.dumps(b04.Route.model_json_schema(), indent=2))

## 2. The router alone
`build_router` returns a function; call it with a state.

In [ ]:
router = b04.build_router(get_cheap_llm())
print(router)
print(router({"messages": [("user", "What is happening with INC-1001?")]}))

## 3. Decisions and reasons for several questions
Watch the two-part question and the ambiguous one.

In [ ]:
router_llm = get_cheap_llm().with_structured_output(b04.Route)
questions = ["How do I set up email on my phone?",
             "What is happening with INC-1001?",
             "Is the VPN down and what is the status of INC-1001?",
             "My laptop is slow",
             "Thanks!"]
for q in questions:
    r = router_llm.invoke([("system", b04.ROUTER_PROMPT), ("user", q)])
    print(f"{r.agent:<13} | {q:<52} | {r.reason}")

## 4. The whole graph, step by step
The router writes `route`; the conditional edge follows it; the specialist adds messages.

In [ ]:
graph = b04.build_graph(llm=get_llm(), router_llm=get_cheap_llm())
with trace_attributes(tags=["notebook-04"]):
    for chunk in graph.stream({"messages": [("user", "What is happening with INC-1001?")]},
                              config=langfuse_config(recursion_limit=20), stream_mode="updates"):
        for node, update in chunk.items():
            shown = {k: (v if k == "route" else f"{len(v)} new message(s)") for k, v in update.items()}
            print(f"after {node}: {shown}")

In [ ]:
flush()